# Day 2: IsoAnnot Practice Session
See also: https://github.com/ConesaLab/SQANTI3/wiki/IsoAnnotLite


# Setup
IsoAnnotLite is included in SQANTI3 QC. For installation instructions, see installation of SQANTI3.

In [2]:
import os
import sys
import subprocess
import pandas as pd

In [5]:
n_cores = 8
data_dir = os.path.abspath(os.path.expanduser("../../data/sqanti3/data"))
solution_dir = os.path.abspath(os.path.expanduser("../../data/sqanti3/output_solution"))
output_dir = os.path.abspath(os.path.expanduser("../output"))
sample_name = "h1_endo_chr8_isotools"
sqanti_path="/home/biouser/envs/sqanti3/"
# reference
ref_genome = f"{data_dir}/GRCh38.primary_assembly.genome_chr8.fa"
ref_gtf = f"{data_dir}/gencode.v39.annotation_chr8.gtf"
tappas_gff3 = f"{data_dir}/human_gencodev39_tappas_annotation.gff3"

# main data
input_gtf = f"{solution_dir}/sqanti_rescue/h1_endo_chr8_isotools/h1_endo_chr8_isotools_rescue_rescued.gtf"
qc_prefix = f"{sample_name}_qc"
qc_dir = f"{output_dir}/sqanti_qc/{sample_name}"
isoannot_prefix = f"{sample_name}_isoannot"
isoannot_dir = f"{output_dir}/sqanti_isoannot/{sample_name}"
isoannot_prefix_man = f"{sample_name}_isoannot_manual"
isoannot_dir_man = f"{output_dir}/sqanti_isoannot_manual/{sample_name}"

# orthogonal data
polyA_motifs = f"{sqanti_path}/data/polyA_motifs/mouse_and_human.polyA_motif.txt"
cage = f"{sqanti_path}/data/ref_TSS_annotation/human.refTSS_v3.1.hg38.bed"
counts = f"{data_dir}/h1_endo_chr8_all_count.txt"
polyA_peaks = f"{data_dir}/atlas.clusters.2.0.GRCh38.96_chr8.bed"
splice_junctions = f"{data_dir}/ENCFF498FDF_ENCFF181VTPSJ.out_chr8.tab"
sr_bam = f"{data_dir}/ENCFF498FDF_ENCFF181VTPAligned.sortedByCoord.out_chr8.bam"


In [6]:
!ls input_gtf

ls: cannot access 'input_gtf': No such file or directory


# IsoAnnot

IsoAnnot is a pipeline for isoform-resolved functional annotation. It uses a variety of predictive algorithms as well as database transference tools on both the RNA- and Protein-sequence level to aggregate different types of functional annotations onto a custom transcriptome.

<img src="../../assets/day2/images/isoannot_diagram.jpg" width=600>

# IsoAnnotLite

As IsoAnnot is currently not easily functional "out-of-the-box", IsoAnnotLite is a more practical option to obtain functional annotations specifically for the species *Homo sapiens*, *Mus musculus*, *Drosophila melanogaster*, *Arabidopsis thaliana*, or *Zea mays*.

It works by transferring the functional annotations from a [**pre-computed IsoAnnot GFF3 file**]([https://app.tappas.org/resources/downloads/gffs/) onto a custom transcriptome.

Annotated GFF3 files can be further analyzed using [tappAS](https://app.tappas.org/).

## Running IsoAnnotLite

In [5]:
# Run IsoAnnotLite

# Build the SQANTI3 QC with IsoAnnotLite command
cmd = [
    "/usr/bin/time", "-v",
    f"{sqanti_path}/sqanti3_qc.py",         # SQANTI3 QC script
    "--isoforms", input_gtf,                # GTF to Quality Control
    "--refGTF", ref_gtf,                    # Reference GTF
    "--refFasta", ref_genome,               # Reference Genome
    "--isoAnnotLite",                       # Run IsoAnnotLite
    "--gff3", tappas_gff3,                  # tappAS gff3 annotations
    "--polyA_motif_list", polyA_motifs,     # PolyA Motif List
    "--polyA_peak", polyA_peaks,            # PolyA Peaks
    "--fl_count", counts,                   # Counts file
    "--coverage", splice_junctions,         # splice junction short-read coverage file (from STAR)
    "--CAGE_peak", cage,                    # CAGE Peak file
    "--SR_bam", sr_bam,                     # Short-read BAM file
    "--output", isoannot_prefix,            # Output Prefix
    "--dir", isoannot_dir,                  # Output Location
    "--cpus", str(n_cores)                  # Number of Threads
]

# Print the command for reference
print("Running command:")
print(" ".join(cmd))

# Run the command
result = subprocess.run(cmd, capture_output=True, text=True)

# Print output and errors
print("Output:")
print(result.stdout)
print("Errors:")
print(result.stderr)

Running command:
/usr/bin/time -v /home/train/longTREC/software/SQANTI3/sqanti3_qc.py --isoforms /home/train/longTREC/day2/data/sqanti3/output_solution/sqanti_rescue/h1_endo_chr8_isotools/h1_endo_chr8_isotools_rescue_rescued.gtf --refGTF /home/train/longTREC/day2/data/sqanti3/data/gencode.v39.annotation_chr8.gtf --refFasta /home/train/longTREC/day2/data/sqanti3/data/GRCh38.primary_assembly.genome_chr8.fa --isoAnnotLite --gff3 /home/train/longTREC/day2/data/sqanti3/data/human_gencodev39_tappas_annotation.gff3 --polyA_motif_list /home/train/longTREC/software/SQANTI3/data/polyA_motifs/mouse_and_human.polyA_motif.txt --polyA_peak /home/train/longTREC/day2/data/sqanti3/data/atlas.clusters.2.0.GRCh38.96_chr8.bed --fl_count /home/train/longTREC/day2/data/sqanti3/data/h1_endo_chr8_all_count.txt --coverage /home/train/longTREC/day2/data/sqanti3/data/ENCFF498FDF_ENCFF181VTPSJ.out_chr8.tab --CAGE_peak /home/train/longTREC/software/SQANTI3/data/ref_TSS_annotation/human.refTSS_v3.1.hg38.bed --SR_ba

In [6]:
%%script echo Skipping directly running IsoAnnotLite
# Run IsoAnnotLite directly to circumvent "-novel" argument

# Build the SQANTI3 QC with IsoAnnotLite command
cmd = [
    "python3",
    f"{sqanti_path}/src/utilities/IsoAnnotLite_SQ3.py",         # IsoAnnotLite script
    f"{qc_dir}/{qc_prefix}_corrected.gtf",
    f"{qc_dir}/{qc_prefix}_classification.txt",
    f"{qc_dir}/{qc_prefix}_junctions.txt",
    "-gff3", tappas_gff3,
    "-o", isoannot_prefix_man,
    "-stdout", isoannot_dir_man
]

# Print the command for reference
print("Running command:")
print(" ".join(cmd))

# Run the command
result = subprocess.run(cmd, capture_output=True, text=True)

# Print output and errors
print("Output:")
print(result.stdout)
print("Errors:")
print(result.stderr)

Skipping directly running IsoAnnotLite


### Investigating results...

#### SOX17

One example of a potentially interesting Gene is SOX17, the "SRY-related HMG-box 17", which shows high expression in the endodermal samples, but next to no expression in h1. This makes sense as SOX17 is known as a transcription factor relevant to endodermal development. In the screenshot below (taken from tappAS), you can see that the Gene expresses two different transcripts, which have varying functional annotations. For example, the PFAM domain PF00505, High mobility group box, is only present in the more highly expressed transcript. On the other hand, the PFAM domain PF12067, which is present in both proteins, is the "central region of eukaryotic SOX17 and 18 transcription factor proteins" - so it seems interesting that the SOX17 gene appears to be expressed, although in relatively small part, as a protein that is missing this name-giving part.

##### Proteins
<img src="../../assets/day2/images/tappAS_SOX17_Gene_Proteins_Visualization.png" >

#### ZNF395

Another potentially interesting Gene could be ZNF395, which is expressed in 3 different transcripts, forming 2 very different proteins.

##### Transcripts
<img src="../../assets/day2/images/tappAS_ZNF395_Gene_Transcripts_Visualization.png" >

##### Proteins
<img src="../../assets/day2/images/tappAS_ZNF395_Gene_Proteins_Visualization.png" >